# Exercise 12 - Residual Networks

The goal of this exercise is to create and run your own residual network starting from a given network with a simple but deep architecture.

We start with a simple but quite deep network of 18 x 3x3 convolutional layers, each with batch normalization in front of a ReLU activation function. Batch normalization should keep the activations and weights under control in this deep network. Run the following code to train this network on CIFAR-10.
- Run this exercise using the **T4 GPU**

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, Activation, MaxPooling2D, Dense, Flatten
from tensorflow.keras.layers import Dropout, BatchNormalization, GlobalAveragePooling2D, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.utils  import to_categorical

# Ensure a clean start
tensorflow.keras.backend.clear_session()

# The data, shuffled and split between train and test sets:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

n_labels = 10

# Convert class vectors to binary class matrices.
y_train = to_categorical(y_train, n_labels)
y_test = to_categorical(y_test, n_labels)

# Normalize the images to have zero mean and values in range [-1,+1]
x_train = x_train.astype('float32') / 255
x_test  = x_test.astype('float32') / 255
mean    = x_train.mean()
x_train = x_train - mean
x_test  = x_test - mean

In [ ]:
import tensorflow
from tensorflow.keras.layers import Input, Dense, Activation, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling2D, add
from tensorflow.keras.models import Model
from tensorflow.keras.utils  import plot_model


tensorflow.keras.backend.clear_session()

def conv_bn_relu(IN, n_features, size, padding='same', strides=(1, 1)):
    L1 = Conv2D(n_features, size, strides=strides, padding=padding, use_bias=False)(IN)
    L2 = BatchNormalization(scale=False)(L1)
    L3 = Activation('relu')(L2)
    return L3

input_img = Input(shape=(32, 32, 3))
IN = input_img

C1 = conv_bn_relu(IN, 16, (3, 3))
C2 = conv_bn_relu(C1, 16, (3, 3))
C3 = conv_bn_relu(C2, 16, (3, 3))
C4 = conv_bn_relu(C3, 16, (3, 3))
C5 = conv_bn_relu(C4, 16, (3, 3))
C6 = conv_bn_relu(C5, 16, (3, 3))

C7 = conv_bn_relu(C6, 32, (3, 3), strides=(2, 2))
C8 = conv_bn_relu(C7, 32, (3, 3))
C9 = conv_bn_relu(C8, 32, (3, 3))
C10 = conv_bn_relu(C9, 32, (3, 3))
C11 = conv_bn_relu(C10, 32, (3, 3))
C12 = conv_bn_relu(C11, 32, (3, 3))

C13 = conv_bn_relu(C12, 64, (3, 3), strides=(2, 2))
C14 = conv_bn_relu(C13, 64, (3, 3))
C15 = conv_bn_relu(C14, 64, (3, 3))
C16 = conv_bn_relu(C15, 64, (3, 3))
C17 = conv_bn_relu(C16, 64, (3, 3))
C18 = conv_bn_relu(C17, 64, (3, 3))

C = Conv2D(10, (1, 1), strides=(1, 1))(C18)
P = GlobalAveragePooling2D(name='avg_pool')(C)
S = Activation('softmax', name='softmax')(P)

model = Model(inputs=input_img, outputs=S)
model.summary()

plot_model(model, "deep_convolution.png", dpi=100, show_shapes=True)

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=32, epochs=10,
          validation_data=(x_test, y_test), shuffle=True);

Now create a residual network based on the model above by adding a shortcut around each pair of convolution filters, as shown on the slides, and compare the results (the accuracy and whether the network is over-fitting the data) to the non-residual version.

In [ ]:
#

#### Solution

If you want some help with the answer, you can look at our answer.  Copy and paste in a new cell to try it out. 

<details>
    <summary> See our answer</summary>
    
    from tensorflow.keras.layers import add
    from tensorflow.keras.utils  import plot_model

    def conv_bn_relu(IN, n_features, size, padding='same', strides=(1, 1)):
        L1 = Conv2D(n_features, size, strides=strides, padding=padding, use_bias=False)(IN)
        L2 = BatchNormalization(scale=False)(L1)
        L3 = Activation('relu')(L2)
        return L3

    def residual_block(IN, n_features, strides=(1, 1)):
        if strides == (1, 1):
            if IN.shape[2] == n_features:
                shortcut = IN
            else:
                shortcut = Conv2D(n_features, (1, 1))(IN)
        else:
            shortcut = MaxPooling2D(pool_size=strides[0], padding='same')(IN)
            shortcut = Conv2D(n_features, (1, 1), strides=(1, 1))(shortcut)
        
        C1 = conv_bn_relu(IN, n_features, (3, 3), strides=strides)
        C2 = conv_bn_relu(C1, n_features, (3, 3))
        return add([C2, shortcut])

    input_img = Input(shape=(32, 32, 3))
    IN = input_img

    B1 = residual_block(IN, 16)
    B2 = residual_block(B1, 16)
    B3 = residual_block(B2, 16)

    B4 = residual_block(B3, 32, strides=(2, 2))
    B5 = residual_block(B4, 32)
    B6 = residual_block(B5, 32)

    B7 = residual_block(B6, 64, strides=(2, 2))
    B8 = residual_block(B7, 64)
    B9 = residual_block(B8, 64)

    C = Conv2D(10, (1, 1), strides=(1, 1))(B9)
    P = GlobalAveragePooling2D(name='avg_pool')(C)
    S = Activation('softmax', name='softmax')(P)

    model = Model(inputs=input_img, outputs=S)
    #model.summary()
    plot_model(model, "mini_resnet.png", dpi=100, show_shapes=True)

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(x_train, y_train,
          batch_size=32, epochs=10,
          validation_data=(x_test, y_test), shuffle=True);
        
</details>
